In [1]:
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.model_selection import train_test_split

This section cleans labelled data and imputes missing values using MICE, implemented by scikit-learn's IterativeImputer. The code:
* Changes True/False labels to 1/0 respectively
* Imputes missing values either by MICE or kNN
* Drops extra columns (Chr, START_POS_REF, END_POS_REF, REF, ALT, REF_MFVdVs, ALT_MFVdVs, Sample_Name)
* And finally saves to a new .txt file

In [3]:
folders = ['syn1', 'syn2', 'syn3', 'syn4', 'syn5', 'real1', 'real2_part1']
methods = ['mice', 'knn']

for name in folders:
    for method in methods:
        data = np.genfromtxt(f'./data/{name}/snv-parse-{name}-labeled.txt', dtype = 'str')
        if method == 'mice':
            imputer = IterativeImputer(sample_posterior = True, initial_strategy = 'median', random_state = 0)
        else:
            imputer = KNNImputer(n_neighbors = 20)
        data[1:, 12:18] = imputer.fit_transform(data[1:, 12:18]).astype('str')
        merged_data = [list(data[0, 8:])]
        for row in data[1:, :]:
            calls = ['1' if x == 'True' else '0' for x in row[8:12]]
            orthogonal = [x[:-2] if x[-2:] == 'e-' else x for x in row[12:18]]
            truth = ['1' if row[-1:] == 'True' else '0']
            merged_data.append(calls + orthogonal + truth)
        with open(f'./data/{name}/snv-parse-{name}-{method}.txt', 'w') as file:
            for row in merged_data:
                file.write(' '.join(row) + '\n')

The next step is to split the data into training and testing sets. real2_part1 is not included in this.

In [4]:
folders = ['syn1', 'syn2', 'syn3', 'syn4', 'syn5', 'real1']
methods = ['mice', 'knn']

for name in folders:
    for method in methods:
        data = np.genfromtxt(f'./data/{name}/snv-parse-{name}-{method}.txt', skip_header = 1, dtype = 'str')
        x, y = data[:, :-1], data[:, -1].astype('int')
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0, stratify = y)
        train_set = np.concatenate([x_train, y_train[:, np.newaxis]], axis = 1)
        with open(f'./data/{name}/snv-parse-{name}-{method}-train.txt', 'w') as file:
            for row in train_set:
                file.write(' '.join(row) + '\n')
        test_set = np.concatenate([x_test, y_test[:, np.newaxis]], axis = 1)
        with open(f'./data/{name}/snv-parse-{name}-{method}-test.txt', 'w') as file:
            for row in test_set:
                file.write(' '.join(row) + '\n')

Finally, we merge the training sets into 1 large training set, and similarly for the testing sets.

In [5]:
subsets = ['train', 'test']
methods = ['mice', 'knn']
folders = ['syn1', 'syn2', 'syn3', 'syn4', 'syn5', 'real1']

for subset in subsets:
    for method in methods:
        with open(f'./data/full/snv-parse-full-{method}-{subset}.txt', 'w') as outfile:
            for name in folders:
                with open(f'./data/{name}/snv-parse-{name}-{method}-{subset}.txt') as infile:
                    for line in infile:
                        outfile.write(line)